# build_graph для Google Colab

1. Установи `rdflib`
2. Загрузи через Upload файлы `ontology.ttl`, `snapshots.json`, `podcasts.json`
3. Запусти построение графа
4. Скачай `knowledge_graph.ttl`


In [ ]:
!pip install -q rdflib


In [ ]:
from google.colab import files
from pathlib import Path
import os

print("Загрузи файлы: ontology.ttl, snapshots.json, podcasts.json")
uploaded = files.upload()

Path("data").mkdir(exist_ok=True)
Path("output").mkdir(exist_ok=True)

for name in uploaded:
    if name == "ontology.ttl":
        pass
    elif name == "snapshots.json":
        os.replace(name, f"data/{name}")
    elif name == "podcasts.json":
        os.replace(name, f"data/{name}")
    else:
        print(f"Файл {name} загружен, но скрипт его не использует")

print("\nТекущая структура:")
for p in Path('.').glob('*'):
    print('-', p)
print('data/:', list(Path('data').glob('*')))


In [ ]:
from __future__ import annotations

import json
import re
from datetime import datetime, timedelta
from pathlib import Path

from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import RDF, RDFS, XSD

ROOT = Path.cwd()
EX = Namespace("http://itmo.ru/spy-multimodal#")

SYNC_WINDOW_SECONDS = 600
BULLISH_LEX = {"bullish", "buy", "strong", "rally", "breakout", "uptrend", "long"}
BEARISH_LEX = {"bearish", "sell", "weak", "drop", "crash", "downtrend", "short", "pullback", "cautious"}


def parse_iso(ts: str) -> datetime:
    return datetime.fromisoformat(ts.replace("Z", "+00:00"))


def detect_polarity(text: str) -> str:
    tokens = set(re.findall(r"[A-Za-z]+", text.lower()))
    bull = len(tokens & BULLISH_LEX)
    bear = len(tokens & BEARISH_LEX)
    if bull > bear:
        return "bullish"
    if bear > bull:
        return "bearish"
    return "neutral"


def find_tickers(text: str) -> list[str]:
    return list({m for m in re.findall(r"\b[A-Z]{1,5}\b", text) if m not in {"I", "A", "CFO", "CFOS"}})


def build_graph(snapshots_path: Path, podcasts_path: Path, ontology_path: Path) -> Graph:
    g = Graph()
    g.parse(ontology_path, format="turtle")
    g.bind("ex", EX)

    snapshots = json.loads(snapshots_path.read_text(encoding="utf-8"))
    podcasts = json.loads(podcasts_path.read_text(encoding="utf-8"))

    ticker_uris: dict[str, URIRef] = {}
    for s in snapshots:
        t = s["ticker"]
        if t not in ticker_uris:
            ticker_uris[t] = EX[f"ticker_{t}"]
            g.add((ticker_uris[t], RDF.type, EX.Ticker))
            g.add((ticker_uris[t], RDFS.label, Literal(t)))

    snap_index: dict[str, dict] = {}
    for s in snapshots:
        snap_uri = EX[s["snapshot_id"]]
        ts_dt = parse_iso(s["timestamp"])
        snap_index[s["snapshot_id"]] = {"ts": ts_dt, "uri": snap_uri, "ticker": s["ticker"]}

        g.add((snap_uri, RDF.type, EX.MarketSnapshot))
        g.add((snap_uri, EX.timestamp, Literal(ts_dt, datatype=XSD.dateTime)))
        g.add((snap_uri, EX.underlyingPrice, Literal(float(s["uPrice"]), datatype=XSD.float)))
        g.add((snap_uri, EX.snapshotVolatility, Literal(float(s["volatility"]), datatype=XSD.float)))
        g.add((snap_uri, EX.onTicker, ticker_uris[s["ticker"]]))

        for kind, klass in (("call", EX.CallOption), ("put", EX.PutOption)):
            for i, opt in enumerate(s.get(kind, [])):
                opt_uri = EX[f"{s['snapshot_id']}_{kind}_{i}"]
                g.add((opt_uri, RDF.type, klass))
                g.add((snap_uri, EX.hasOption, opt_uri))
                g.add((opt_uri, EX.onTicker, ticker_uris[s["ticker"]]))

                for prop, key, dt in [
                    (EX.lastPrice, "last", XSD.float),
                    (EX.bid, "bid", XSD.float),
                    (EX.ask, "ask", XSD.float),
                    (EX.bidSize, "bidSize", XSD.integer),
                    (EX.askSize, "askSize", XSD.integer),
                    (EX.ivOption, "volatility", XSD.float),
                    (EX.intrinsicValue, "intrinsicValue", XSD.float),
                    (EX.extrinsicValue, "extrinsicValue", XSD.float),
                    (EX.optionId, "OptionId", XSD.string),
                    (EX.strike, "strike", XSD.float),
                ]:
                    if key in opt and opt[key] is not None:
                        g.add((opt_uri, prop, Literal(opt[key], datatype=dt)))

                if "expiry" in opt:
                    g.add((opt_uri, EX.expiry, Literal(opt["expiry"], datatype=XSD.date)))

                greek = opt.get("greek", {})
                for prop, key in [
                    (EX.gamma, "gamma"),
                    (EX.delta, "delta"),
                    (EX.theta, "theta"),
                    (EX.vega, "vega"),
                    (EX.rho, "rho"),
                ]:
                    if key in greek and greek[key] is not None:
                        g.add((opt_uri, prop, Literal(float(greek[key]), datatype=XSD.float)))

    seg_index: list[dict] = []
    for pod in podcasts:
        pod_uri = EX[pod["podcast_id"]]
        g.add((pod_uri, RDF.type, EX.Podcast))
        g.add((pod_uri, RDFS.label, Literal(pod["title"])))
        rec_dt = parse_iso(pod["recorded_at"])
        g.add((pod_uri, EX.timestamp, Literal(rec_dt, datatype=XSD.dateTime)))

        for seg in pod["segments"]:
            seg_uri = EX[f"{pod['podcast_id']}_seg_{seg['id']}"]
            seg_dt = rec_dt + timedelta(seconds=float(seg["start"]))
            polarity = detect_polarity(seg["text"])
            tickers_in_seg = find_tickers(seg["text"])

            g.add((seg_uri, RDF.type, EX.PodcastSegment))
            g.add((seg_uri, EX.partOfPodcast, pod_uri))
            g.add((seg_uri, EX.segmentIndex, Literal(int(seg["id"]), datatype=XSD.integer)))
            g.add((seg_uri, EX.startTime, Literal(float(seg["start"]), datatype=XSD.float)))
            g.add((seg_uri, EX.endTime, Literal(float(seg["end"]), datatype=XSD.float)))
            g.add((seg_uri, EX.duration, Literal(float(seg["end"] - seg["start"]), datatype=XSD.float)))
            g.add((seg_uri, EX.transcriptText, Literal(seg["text"].strip(), datatype=XSD.string)))
            g.add((seg_uri, EX.timestamp, Literal(seg_dt, datatype=XSD.dateTime)))
            g.add((seg_uri, EX.polarity, Literal(polarity, datatype=XSD.string)))

            for opt_field in ("avg_logprob", "compression_ratio", "no_speech_prob"):
                if opt_field in seg:
                    prop = {
                        "avg_logprob": EX.avgLogprob,
                        "compression_ratio": EX.compressionRatio,
                        "no_speech_prob": EX.noSpeechProb,
                    }[opt_field]
                    g.add((seg_uri, prop, Literal(float(seg[opt_field]), datatype=XSD.float)))

            for t in tickers_in_seg:
                if t in ticker_uris:
                    g.add((seg_uri, EX.mentionsTicker, ticker_uris[t]))

            for snap_id, info in snap_index.items():
                if info["ticker"] not in tickers_in_seg:
                    continue
                if abs((info["ts"] - seg_dt).total_seconds()) <= SYNC_WINDOW_SECONDS:
                    g.add((seg_uri, EX.coOccursWith, info["uri"]))

            seg_index.append({
                "uri": seg_uri,
                "ts": seg_dt,
                "polarity": polarity,
                "tickers": tickers_in_seg,
                "text": seg["text"],
            })

    for snap_id, info in snap_index.items():
        related = [
            s for s in seg_index
            if info["ticker"] in s["tickers"] and abs((info["ts"] - s["ts"]).total_seconds()) <= SYNC_WINDOW_SECONDS
        ]
        bull = sum(1 for s in related if s["polarity"] == "bullish")
        bear = sum(1 for s in related if s["polarity"] == "bearish")

        if bull > bear:
            decision, dec_uri = "BUY", EX.Buy
        elif bear > bull:
            decision, dec_uri = "SELL", EX.Sell
        else:
            decision, dec_uri = "HOLD", EX.Hold

        confidence = (abs(bull - bear) / max(bull + bear, 1)) if related else 0.0

        sig_uri = EX[f"signal_{snap_id}"]
        g.add((sig_uri, RDF.type, EX.TradingSignal))
        g.add((sig_uri, EX.appliesTo, info["uri"]))
        g.add((sig_uri, EX.onTicker, ticker_uris[info["ticker"]]))
        g.add((sig_uri, EX.hasDecision, dec_uri))
        g.add((sig_uri, EX.decisionLabel, Literal(decision, datatype=XSD.string)))
        g.add((sig_uri, EX.horizonMin, Literal(60, datatype=XSD.integer)))
        g.add((sig_uri, EX.confidence, Literal(round(confidence, 3), datatype=XSD.float)))
        g.add((sig_uri, EX.timestamp, Literal(info["ts"], datatype=XSD.dateTime)))
        g.add((sig_uri, EX.rationale, Literal(f"bullish={bull}, bearish={bear}, related_segments={len(related)}", datatype=XSD.string)))

        for s in related:
            g.add((sig_uri, EX.evidencedBy, s["uri"]))

    return g


QUERIES = [
    {
        "title": "Q1. Все BUY-сигналы с обоснованием из подкаста",
        "nl": "Найти торговые сигналы с решением BUY и список подкаст-сегментов, послуживших обоснованием.",
        "sparql": """
PREFIX ex: <http://itmo.ru/spy-multimodal#>
SELECT ?signal ?decision ?confidence ?segment ?text WHERE {
  ?signal a ex:TradingSignal ;
          ex:decisionLabel ?decision ;
          ex:confidence ?confidence ;
          ex:evidencedBy ?segment .
  ?segment ex:transcriptText ?text .
  FILTER (?decision = \"BUY\")
} ORDER BY DESC(?confidence)
"""
    },
    {
        "title": "Q2. Распределение полярности упоминаний SPY",
        "nl": "Сколько подкаст-сегментов с упоминанием SPY имеют каждую полярность.",
        "sparql": """
PREFIX ex: <http://itmo.ru/spy-multimodal#>
SELECT ?polarity (COUNT(?seg) AS ?cnt) WHERE {
  ?seg a ex:PodcastSegment ;
       ex:mentionsTicker ex:ticker_SPY ;
       ex:polarity ?polarity .
} GROUP BY ?polarity ORDER BY DESC(?cnt)
"""
    },
    {
        "title": "Q3. Снимки с со-встречающимся аудио-обсуждением",
        "nl": "Какие рыночные снимки SPY имеют связанные подкаст-сегменты, обсуждающие тот же актив в окне ±10 минут.",
        "sparql": """
PREFIX ex: <http://itmo.ru/spy-multimodal#>
SELECT ?snapshot ?uPrice ?segText WHERE {
  ?snapshot a ex:MarketSnapshot ;
            ex:underlyingPrice ?uPrice .
  ?seg ex:coOccursWith ?snapshot ;
       ex:transcriptText ?segText .
}
"""
    },
    {
        "title": "Q4. Колл-опционы с дельтой ATM-зоны (0.4 ≤ delta ≤ 0.6)",
        "nl": "Перечислить колл-опционы с дельтой ATM (0.4 ≤ delta ≤ 0.6) и их страйки.",
        "sparql": """
PREFIX ex: <http://itmo.ru/spy-multimodal#>
SELECT ?option ?strike ?delta ?last WHERE {
  ?option a ex:CallOption ;
          ex:strike ?strike ;
          ex:delta ?delta ;
          ex:lastPrice ?last .
  FILTER (?delta >= 0.4 && ?delta <= 0.6)
} ORDER BY ?strike
"""
    },
    {
        "title": "Q5. Полный мультимодальный контекст BUY-сигнала",
        "nl": "Для каждого BUY-сигнала: цена базового актива в момент сигнала и тексты обосновывающих подкаст-сегментов.",
        "sparql": """
PREFIX ex: <http://itmo.ru/spy-multimodal#>
SELECT ?signal ?uPrice ?segText WHERE {
  ?signal a ex:TradingSignal ;
          ex:hasDecision ex:Buy ;
          ex:appliesTo ?snap ;
          ex:evidencedBy ?seg .
  ?snap ex:underlyingPrice ?uPrice .
  ?seg ex:transcriptText ?segText .
}
"""
    },
]


def run_queries(g: Graph) -> None:
    for q in QUERIES:
        print("=" * 78)
        print(q["title"])
        print(f"NL: {q['nl']}")
        print("-" * 78)
        rows = list(g.query(q["sparql"]))
        if not rows:
            print("(нет результатов)")
            continue
        for row in rows:
            print(" | ".join(str(v) for v in row))
        print(f"Всего строк: {len(rows)}")


def main():
    ontology = ROOT / "ontology.ttl"
    snapshots = ROOT / "data" / "snapshots.json"
    podcasts = ROOT / "data" / "podcasts.json"
    out_dir = ROOT / "output"
    out_dir.mkdir(exist_ok=True)
    out_ttl = out_dir / "knowledge_graph.ttl"

    for path in [ontology, snapshots, podcasts]:
        if not path.exists():
            raise FileNotFoundError(f"Не найден файл: {path}")

    print("Building knowledge graph...")
    g = build_graph(snapshots, podcasts, ontology)
    g.serialize(out_ttl, format="turtle")
    print(f"Triples: {len(g)}")
    print(f"Saved: {out_ttl}")

    print("\nRunning SPARQL queries...\n")
    run_queries(g)

main()


In [ ]:
from google.colab import files
files.download('output/knowledge_graph.ttl')
